# Лабораторне заняття №0
## Налаштування робочого середовища та основи роботи з pandas
Студент: Абдулханов Абдул-Рахім, група ФеП-33

### 1. Порівняння середовищ розробки
| Характеристика | PyCharm | Jupyter Notebook | Google Colab |
| --- | --- | --- | --- |
| Потрібне встановлення | Так | Так | Ні |
| Потрібен Python на ПК | Так | Так | Ні |
| Основний формат файлів | .py | .ipynb | .ipynb |
| Виконання коду | Повний скрипт | По окремих комірках | По окремих комірках |
| Спільний доступ | Через VCS (Git) | Локальний файл | Пряме посилання в хмарі |

In [ ]:
# імпортуємо бібліотеки для роботи з даними
import os
import pandas as pd
import matplotlib.pyplot as plt
print('pandas version:', pd.__version__)

### 2. Зчитування даних та дослідження параметрів
Датасет iris1.csv містить 150 спостережень за трьома видами ірисів (Setosa, Versicolor, Virginica) з чотирма морфологічними ознаками (довжина та ширина чашолистка і пелюстки).

In [ ]:
# зчитуємо дані та досліджуємо параметри
df = pd.read_csv('task/iris1.csv')
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print('index:', df.index)
print('dtypes:\n', df.dtypes)
df.info()
print('\nhead:\n', df.head(3))
print('\ntail:\n', df.tail(3))
print('\nsample:\n', df.sample(3, random_state=42))

### 3. Створення нових колонок, вибірка, індексація, drop та query

In [ ]:
# створення колонки суми двох колонок та їх співвідношення
df['total_length'] = df['sepal.length'] + df['petal.length']
df['sepal_ratio'] = (df['sepal.length'] / df['sepal.width']).round(2)

# вибірка loc та iloc
print('loc subset:\n', df.loc[:2, ['sepal.length', 'variety']])
print('\niloc subset:\n', df.iloc[:2, [0, 2]])

# встановлення індексу set_index та видалення стовпця drop
df_indexed = df.set_index('variety')
print('\nset_index preview:\n', df_indexed.head(2))
df_dropped = df.drop(columns=['total_length'])
print('columns after drop:', df_dropped.columns.tolist())

# умовна фільтрація та метод query
print('setosa count:', len(df[df['variety'] == 'Setosa']))
query_res = df.query('`sepal.length` > 6.0 and variety == "Virginica"')
print('query count (>6.0 and Virginica):', len(query_res))

### 4. Summary Functions, Maps та Lambda вирази

In [ ]:
# частоти, унікальні, сума, середнє
print('value_counts:\n', df['variety'].value_counts())
print('unique:', df['variety'].unique())
print('sum petal length:', round(df['petal.length'].sum(), 2))
print('mean sepal length:', round(df['sepal.length'].mean(), 2))

# map зі словником та лямбда функцією
species_code = {'Setosa': 0, 'Versicolor': 1, 'Virginica': 2}
df['species_id'] = df['variety'].map(species_code)
df['variety_upper'] = df['variety'].map(lambda x: str(x).upper())
df[['variety', 'species_id', 'variety_upper']].head(3)

### 5. Групування, сортування та вплив параметрів

In [ ]:
# сортування ascending=True та False
print('sort ascending:\n', df.sort_values(by='sepal.length', ascending=True)[['sepal.length', 'variety']].head(2))
print('sort descending:\n', df.sort_values(by='sepal.length', ascending=False)[['sepal.length', 'variety']].head(2))

# групування за 1 та 2 колонками
df['sepal_cat'] = pd.qcut(df['sepal.length'], q=3, labels=['short', 'medium', 'long'])
print('\ngrouped by 1 col:\n', df.groupby('variety')['sepal.length'].mean())
print('\ngrouped by 2 cols:\n', df.groupby(['variety', 'sepal_cat'], observed=False)['petal.length'].mean())

# демонстрація впливу параметрів (2 приклади)
# приклад 1: sort_values ascending=True vs False (впливає на порядок вибірки)
# приклад 2: groupby as_index=True (повертає Series з мультиіндексом) vs as_index=False (повертає DataFrame)
print('\nas_index=True type:', type(df.groupby('variety', as_index=True)['petal.length'].mean()))
print('as_index=False type:', type(df.groupby('variety', as_index=False)['petal.length'].mean()))

### 6. Робота з датами та складніші запити

In [ ]:
# датасет iris1.csv не містить часових ознак, демонструємо розрахунок кількості місяців до сьогодні
sample_dates = pd.Series(['2023-01-15', '2024-06-01', '2025-09-20'])
parsed = pd.to_datetime(sample_dates)
now = pd.Timestamp.now()
months_to_now = (now.year - parsed.dt.year) * 12 + (now.month - parsed.dt.month)
print('months to today:\n', pd.DataFrame({'date': sample_dates, 'months': months_to_now}))

# крос-табуляція та відбір топ-значень
print('\ncrosstab:\n', pd.crosstab(df['variety'], df['sepal_cat']))

### 7. Візуалізація даних

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for species, grp in df.groupby('variety'):
    ax[0].scatter(grp['sepal.length'], grp['petal.length'], label=species, alpha=0.8)
ax[0].set_xlabel('sepal length (cm)')
ax[0].set_ylabel('petal length (cm)')
ax[0].set_title('sepal vs petal length')
ax[0].legend()
ax[0].grid(True, linestyle='--', alpha=0.5)

df['petal.width'].hist(ax=ax[1], bins=15, color='#2b5c8f', edgecolor='black')
ax[1].set_xlabel('petal width (cm)')
ax[1].set_ylabel('count')
ax[1].set_title('petal width distribution')
ax[1].grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()